# Stage 2: Feature Extraction and Unsupervised Analysis

This notebook covers:
1. **Understanding Feature Representations** — ArcFace (InsightFace) and VGG19 architecture inspection
2. **Clustering Clean Data** — K-means on gallery + clean probe images
3. **Clustering Noisy Data** — K-means on probe images under varying conditions

**Setup:** Upload your dataset zip to Colab or mount Google Drive before running.

---
## 0. Installation & Imports

In [ ]:
# Install InsightFace and ONNX
!pip install -q insightface onnx onnxruntime-gpu

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import cv2
from PIL import Image

import torch
import torchvision.models as models
import torchvision.transforms as transforms

import insightface
from insightface.app import FaceAnalysis

import onnx

from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment

print('All imports successful.')

---
## 0.1 Mount Google Drive & Set Dataset Path

Upload your dataset to Google Drive with this structure:
```
Dataset/
  KimK/
    KimGallery/          <- 12 clean gallery images
    KimProbe/
      KimProbe_Clean/    <- clean probe images
      KimProbe_Side/     <- side angle
      KimProbe_Sunglass/ <- occlusion
      KimProbe_exHigh/   <- high exposure
      KimProbe_exLow/    <- low exposure
      KimProbe_lightLow/ <- low lighting
  LeoDicaprio/
    DicaprioGallery/
    DicaprioProbe/
      ...
  MeganFox/
    FoxGallery/
    FoxProbe/
      ...
  TaylorSwift/
    SwiftGallery/
    SwiftProbe/
      ...
  ... (up to 10 identities)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- EDIT THIS PATH to match your Drive folder ----
DATASET_ROOT = '/content/drive/MyDrive/Data Science Pics'

# Verify identities found
identities = sorted([d for d in os.listdir(DATASET_ROOT)
                      if os.path.isdir(os.path.join(DATASET_ROOT, d))])
NUM_IDENTITIES = len(identities)
print(f'Found {NUM_IDENTITIES} identities: {identities}')

---
## 0.2 Dataset Loading Utilities

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(folder):
    """Return sorted list of image file paths in a folder."""
    files = []
    for f in sorted(os.listdir(folder)):
        if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
            files.append(os.path.join(folder, f))
    return files


def find_gallery_and_probe_dirs(identity_dir):
    """
    Given an identity folder (e.g., 'KimK/'), find the gallery dir and probe sub-dirs.
    Returns: (gallery_dir, {condition_name: probe_condition_dir})
    """
    gallery_dir = None
    probe_dirs = {}
    for item in sorted(os.listdir(identity_dir)):
        full = os.path.join(identity_dir, item)
        if not os.path.isdir(full):
            continue
        if 'gallery' in item.lower():
            gallery_dir = full
        elif 'probe' in item.lower():
            # This is the probe parent — look inside for condition sub-dirs
            for sub in sorted(os.listdir(full)):
                sub_full = os.path.join(full, sub)
                if os.path.isdir(sub_full):
                    # Extract condition name (last part after final underscore)
                    condition = sub.split('_')[-1]  # e.g., 'Clean', 'Side', etc.
                    probe_dirs[condition] = sub_full
    return gallery_dir, probe_dirs


# Build the full dataset structure
dataset = {}  # {identity: {'gallery': dir, 'probe': {condition: dir}}}
for identity in identities:
    identity_dir = os.path.join(DATASET_ROOT, identity)
    gallery_dir, probe_dirs = find_gallery_and_probe_dirs(identity_dir)
    dataset[identity] = {'gallery': gallery_dir, 'probe': probe_dirs}
    n_gallery = len(list_images(gallery_dir)) if gallery_dir else 0
    probe_counts = {k: len(list_images(v)) for k, v in probe_dirs.items()}
    print(f'{identity}: gallery={n_gallery}, probe conditions={probe_counts}')

# Collect all probe conditions across all identities
all_conditions = set()
for info in dataset.values():
    all_conditions.update(info['probe'].keys())
all_conditions = sorted(all_conditions)
print(f'\nProbe conditions: {all_conditions}')

---
# 1. Understanding Feature Representations

## 1.1 ArcFace (InsightFace) — Setup & Architecture

In [ ]:
# Initialize InsightFace FaceAnalysis
# This downloads the 'buffalo_l' model pack (detection + recognition)
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

# Show which models are loaded
print('Loaded InsightFace models:')
for name, model in app.models.items():
    print(f'  {name}: {type(model).__name__}')

### Inspect the ONNX model graph (recognition model)

In [ ]:
# Find the recognition ONNX model path
insightface_model_dir = os.path.expanduser('~/.insightface/models/buffalo_l')
onnx_path = os.path.join(insightface_model_dir, 'w600k_r50.onnx')

if os.path.exists(onnx_path):
    model_onnx = onnx.load(onnx_path)
    print('=== ArcFace (w600k_r50) ONNX Graph ===')
    print(onnx.helper.printable_graph(model_onnx.graph))
else:
    print(f'ONNX model not found at {onnx_path}')
    print('Available files:', os.listdir(insightface_model_dir))

### How InsightFace works (detection → alignment → embedding)

**Write your description here:**

- **Detection:** ...
- **Alignment:** ...
- **Embedding:** ...
- **ArcFace architecture:** ...
- **Output:** 512-dimensional normalized embedding vector

## 1.2 VGG19 — Setup & Architecture

In [ ]:
# Load pretrained VGG19 and remove the final classifier layer
# We extract features from the first FC layer (fc1 → 4096-dim)
vgg19_full = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
vgg19_full.eval()

# The classifier is: [fc1(4096), relu, dropout, fc2(4096), relu, dropout, fc3(1000)]
# We keep up to and including fc1 (index 0) to get 4096-dim features
vgg19_feature_extractor = torch.nn.Sequential(
    vgg19_full.features,
    vgg19_full.avgpool,
    torch.nn.Flatten(),
    vgg19_full.classifier[0],   # Linear(25088, 4096)
    vgg19_full.classifier[1],   # ReLU
)
vgg19_feature_extractor.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vgg19_feature_extractor = vgg19_feature_extractor.to(device)
print(f'VGG19 feature extractor ready on {device}')

# VGG19 preprocessing
vgg_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Print VGG19 architecture
print(vgg19_full)

### VGG19 Architecture Description

**Write your description here:**

- **Architecture:** ...
- **Feature extraction:** ...
- **Output:** 4096-dimensional feature vector from the first fully-connected layer

---
# Feature Extraction Functions

In [ ]:
def extract_arcface_embedding(image_path, app):
    """
    Extract 512-dim ArcFace embedding using InsightFace.
    Returns embedding vector or None if no face detected.
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f'  Warning: could not read {image_path}')
        return None
    faces = app.get(img)
    if len(faces) == 0:
        print(f'  Warning: no face detected in {image_path}')
        return None
    # Take the face with highest detection score
    face = max(faces, key=lambda f: f.det_score)
    return face.embedding  # 512-dim numpy array


def extract_vgg_features(image_path, model, transform, device):
    """
    Extract 4096-dim VGG19 feature vector.
    """
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = model(tensor)
    return features.cpu().numpy().flatten()  # 4096-dim


def extract_all_features(dataset, app, vgg_model, vgg_transform, device):
    """
    Extract ArcFace and VGG features for all images.

    Returns:
      features: dict with structure:
        {identity: {
            'gallery': {'arcface': [vectors], 'vgg': [vectors], 'paths': [paths]},
            'probe': {
                condition: {'arcface': [...], 'vgg': [...], 'paths': [...]}
            }
        }}
    """
    features = {}
    for identity, info in dataset.items():
        print(f'\nProcessing {identity}...')
        features[identity] = {'gallery': {'arcface': [], 'vgg': [], 'paths': []},
                              'probe': {}}

        # Gallery
        gallery_images = list_images(info['gallery'])
        print(f'  Gallery: {len(gallery_images)} images')
        for img_path in gallery_images:
            arc_emb = extract_arcface_embedding(img_path, app)
            vgg_feat = extract_vgg_features(img_path, vgg_model, vgg_transform, device)
            if arc_emb is not None:
                features[identity]['gallery']['arcface'].append(arc_emb)
                features[identity]['gallery']['vgg'].append(vgg_feat)
                features[identity]['gallery']['paths'].append(img_path)

        # Probe conditions
        for condition, cond_dir in info['probe'].items():
            probe_images = list_images(cond_dir)
            print(f'  Probe [{condition}]: {len(probe_images)} images')
            features[identity]['probe'][condition] = {'arcface': [], 'vgg': [], 'paths': []}
            for img_path in probe_images:
                arc_emb = extract_arcface_embedding(img_path, app)
                vgg_feat = extract_vgg_features(img_path, vgg_model, vgg_transform, device)
                if arc_emb is not None:
                    features[identity]['probe'][condition]['arcface'].append(arc_emb)
                    features[identity]['probe'][condition]['vgg'].append(vgg_feat)
                    features[identity]['probe'][condition]['paths'].append(img_path)

    return features

print('Feature extraction functions defined.')

In [ ]:
# Run feature extraction on the full dataset
features = extract_all_features(dataset, app, vgg19_feature_extractor, vgg_transform, device)

# Quick summary
for identity in identities:
    n_gal = len(features[identity]['gallery']['arcface'])
    probe_summary = {k: len(v['arcface']) for k, v in features[identity]['probe'].items()}
    print(f'{identity}: gallery={n_gal}, probe={probe_summary}')

---
# 2. Clustering Clean Data

Combine gallery images (all) + clean probe images, then apply k-means with k = number of identities.

In [ ]:
def build_feature_matrix(features, identities, probe_conditions):
    """
    Build feature matrices and label vectors from gallery + specified probe conditions.

    Args:
        features: the full features dict
        identities: list of identity names
        probe_conditions: list of condition names to include (e.g., ['Clean'] or all)

    Returns:
        arcface_X, vgg_X: feature matrices (N x dim)
        labels: integer labels (N,)
        label_map: {identity_name: int_label}
        sources: list of (identity, 'gallery'/'probe_condition') per sample
    """
    arcface_list, vgg_list, labels_list, sources_list = [], [], [], []
    label_map = {name: i for i, name in enumerate(identities)}

    for identity in identities:
        label = label_map[identity]
        info = features[identity]

        # Gallery
        for arc, vgg in zip(info['gallery']['arcface'], info['gallery']['vgg']):
            arcface_list.append(arc)
            vgg_list.append(vgg)
            labels_list.append(label)
            sources_list.append((identity, 'gallery'))

        # Probe (selected conditions)
        for condition in probe_conditions:
            if condition in info['probe']:
                for arc, vgg in zip(info['probe'][condition]['arcface'],
                                    info['probe'][condition]['vgg']):
                    arcface_list.append(arc)
                    vgg_list.append(vgg)
                    labels_list.append(label)
                    sources_list.append((identity, f'probe_{condition}'))

    arcface_X = np.array(arcface_list)
    vgg_X = np.array(vgg_list)
    labels = np.array(labels_list)
    return arcface_X, vgg_X, labels, label_map, sources_list

print('build_feature_matrix defined.')

In [ ]:
def cluster_and_evaluate(X, true_labels, n_clusters, label_map, model_name=''):
    """
    Run k-means and evaluate clustering accuracy using Hungarian matching.

    Returns:
        accuracy, cluster_labels, best_mapping
    """
    kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=42)
    cluster_labels = kmeans.fit_predict(X)

    # Build cost matrix for Hungarian algorithm to find best cluster→label mapping
    cost_matrix = np.zeros((n_clusters, n_clusters))
    for cluster_id in range(n_clusters):
        for true_id in range(n_clusters):
            # Count how many samples in this cluster have this true label
            cost_matrix[cluster_id, true_id] = -np.sum(
                (cluster_labels == cluster_id) & (true_labels == true_id)
            )

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    best_mapping = {row: col for row, col in zip(row_ind, col_ind)}

    # Map cluster labels to true labels
    mapped_labels = np.array([best_mapping[c] for c in cluster_labels])
    accuracy = accuracy_score(true_labels, mapped_labels)

    inv_label_map = {v: k for k, v in label_map.items()}
    print(f'\n{model_name} K-Means Clustering Results:')
    print(f'  Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)')
    print(f'  Cluster→Identity mapping:')
    for c, t in sorted(best_mapping.items()):
        print(f'    Cluster {c} → {inv_label_map[t]}')

    return accuracy, cluster_labels, mapped_labels, best_mapping

print('cluster_and_evaluate defined.')

### 2.1 K-Means on Clean Data (Gallery + Clean Probe)

In [ ]:
# Build clean data: gallery + Clean probe only
arc_clean, vgg_clean, labels_clean, label_map, sources_clean = build_feature_matrix(
    features, identities, probe_conditions=['Clean']
)
print(f'Clean dataset: {arc_clean.shape[0]} samples, {NUM_IDENTITIES} identities')
print(f'  ArcFace features: {arc_clean.shape}')
print(f'  VGG features: {vgg_clean.shape}')

In [ ]:
# K-Means clustering — ArcFace clean
acc_arc_clean, clust_arc_clean, mapped_arc_clean, _ = cluster_and_evaluate(
    arc_clean, labels_clean, NUM_IDENTITIES, label_map, model_name='ArcFace (Clean)'
)

# K-Means clustering — VGG clean
acc_vgg_clean, clust_vgg_clean, mapped_vgg_clean, _ = cluster_and_evaluate(
    vgg_clean, labels_clean, NUM_IDENTITIES, label_map, model_name='VGG19 (Clean)'
)

### 2.2 Confusion Matrices — Clean Data

In [ ]:
def plot_confusion_matrices(true_labels, mapped_arc, mapped_vgg, label_map, title_suffix=''):
    inv_map = {v: k for k, v in label_map.items()}
    names = [inv_map[i] for i in range(len(label_map))]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, mapped, title in zip(axes,
                                  [mapped_arc, mapped_vgg],
                                  [f'ArcFace {title_suffix}', f'VGG19 {title_suffix}']):
        cm = confusion_matrix(true_labels, mapped)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=names, yticklabels=names, ax=ax)
        ax.set_title(title)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

    plt.tight_layout()
    plt.show()


plot_confusion_matrices(labels_clean, mapped_arc_clean, mapped_vgg_clean,
                        label_map, title_suffix='— Clean Data')

### 2.3 PCA Visualization — Clean Data

In [ ]:
def plot_pca_clusters(X, labels, label_map, title=''):
    inv_map = {v: k for k, v in label_map.items()}
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)

    plt.figure(figsize=(10, 7))
    for label_id in sorted(label_map.values()):
        mask = labels == label_id
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                    label=inv_map[label_id], alpha=0.7, s=50)
    plt.legend()
    plt.title(title)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
    plt.grid(True, alpha=0.3)
    plt.show()


plot_pca_clusters(arc_clean, labels_clean, label_map, 'ArcFace Embeddings — Clean (PCA)')
plot_pca_clusters(vgg_clean, labels_clean, label_map, 'VGG19 Features — Clean (PCA)')

### 2.4 Comparison: ArcFace vs VGG on Clean Data

**Write your analysis here:**

- ArcFace accuracy: ...
- VGG19 accuracy: ...
- Which model performs better and why?

In [ ]:
print('=== Clean Data Summary ===')
print(f'ArcFace accuracy: {acc_arc_clean:.4f}')
print(f'VGG19   accuracy: {acc_vgg_clean:.4f}')

---
# 3. Clustering Noisy Data

Now include probe images from all conditions (Side, Sunglass, exHigh, exLow, lightLow).

### 3.1 K-Means on Noisy Data (Gallery + All Probe Conditions)

In [ ]:
# Build noisy data: gallery + all probe conditions
noisy_conditions = [c for c in all_conditions if c != 'Clean']
arc_noisy, vgg_noisy, labels_noisy, _, sources_noisy = build_feature_matrix(
    features, identities, probe_conditions=noisy_conditions
)
print(f'Noisy dataset: {arc_noisy.shape[0]} samples')
print(f'  Conditions included: {noisy_conditions}')

In [ ]:
# K-Means — ArcFace noisy
acc_arc_noisy, clust_arc_noisy, mapped_arc_noisy, _ = cluster_and_evaluate(
    arc_noisy, labels_noisy, NUM_IDENTITIES, label_map, model_name='ArcFace (Noisy)'
)

# K-Means — VGG noisy
acc_vgg_noisy, clust_vgg_noisy, mapped_vgg_noisy, _ = cluster_and_evaluate(
    vgg_noisy, labels_noisy, NUM_IDENTITIES, label_map, model_name='VGG19 (Noisy)'
)

In [ ]:
# Confusion matrices — noisy
plot_confusion_matrices(labels_noisy, mapped_arc_noisy, mapped_vgg_noisy,
                        label_map, title_suffix='— Noisy Data')

In [ ]:
# PCA visualization — noisy
plot_pca_clusters(arc_noisy, labels_noisy, label_map, 'ArcFace Embeddings — Noisy (PCA)')
plot_pca_clusters(vgg_noisy, labels_noisy, label_map, 'VGG19 Features — Noisy (PCA)')

### 3.2 Per-Condition Accuracy Breakdown

In [ ]:
# Evaluate clustering accuracy per condition individually
print(f'{"Condition":<15} {"ArcFace Acc":<15} {"VGG19 Acc":<15}')
print('-' * 45)

condition_results = {}
for condition in all_conditions:
    arc_c, vgg_c, labels_c, _, _ = build_feature_matrix(
        features, identities, probe_conditions=[condition]
    )
    if len(labels_c) == 0:
        continue

    acc_a, _, _, _ = cluster_and_evaluate(arc_c, labels_c, NUM_IDENTITIES, label_map, model_name='')
    acc_v, _, _, _ = cluster_and_evaluate(vgg_c, labels_c, NUM_IDENTITIES, label_map, model_name='')
    condition_results[condition] = (acc_a, acc_v)
    print(f'{condition:<15} {acc_a:<15.4f} {acc_v:<15.4f}')

In [ ]:
# Bar chart: per-condition accuracy comparison
conditions = list(condition_results.keys())
arc_accs = [condition_results[c][0] for c in conditions]
vgg_accs = [condition_results[c][1] for c in conditions]

x = np.arange(len(conditions))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, arc_accs, width, label='ArcFace', color='steelblue')
ax.bar(x + width/2, vgg_accs, width, label='VGG19', color='coral')
ax.set_ylabel('Clustering Accuracy')
ax.set_title('K-Means Accuracy by Probe Condition')
ax.set_xticks(x)
ax.set_xticklabels(conditions, rotation=30)
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Feature Distribution Comparison — Two Selected Identities

Compare clean vs. noisy feature distributions for two identities to visualize how conditions affect embeddings.

In [ ]:
# Pick two identities for detailed comparison
id1, id2 = identities[0], identities[1]
print(f'Comparing feature distributions for: {id1} and {id2}')


def plot_identity_condition_pca(features, identity, label_map, model_key='arcface', title_prefix=''):
    """
    PCA plot showing gallery vs each probe condition for a single identity.
    """
    all_vecs = []
    all_labels = []

    # Gallery
    gal_vecs = features[identity]['gallery'][model_key]
    all_vecs.extend(gal_vecs)
    all_labels.extend(['Gallery'] * len(gal_vecs))

    # Each probe condition
    for condition in sorted(features[identity]['probe'].keys()):
        cond_vecs = features[identity]['probe'][condition][model_key]
        all_vecs.extend(cond_vecs)
        all_labels.extend([condition] * len(cond_vecs))

    if len(all_vecs) < 3:
        print(f'Not enough data for {identity}')
        return

    X = np.array(all_vecs)
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)

    plt.figure(figsize=(10, 7))
    unique_labels = ['Gallery'] + sorted(set(all_labels) - {'Gallery'})
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

    for lbl, color in zip(unique_labels, colors):
        mask = np.array([l == lbl for l in all_labels])
        marker = 's' if lbl == 'Gallery' else 'o'
        size = 80 if lbl == 'Gallery' else 50
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                    label=lbl, color=color, marker=marker, s=size, alpha=0.7)

    plt.legend()
    plt.title(f'{title_prefix} {identity} — Gallery vs Probe Conditions')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.grid(True, alpha=0.3)
    plt.show()


# ArcFace distributions
plot_identity_condition_pca(features, id1, label_map, 'arcface', 'ArcFace')
plot_identity_condition_pca(features, id2, label_map, 'arcface', 'ArcFace')

# VGG distributions
plot_identity_condition_pca(features, id1, label_map, 'vgg', 'VGG19')
plot_identity_condition_pca(features, id2, label_map, 'vgg', 'VGG19')

### 3.4 Cosine Similarity Analysis — Clean vs Noisy

In [ ]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))


def compute_avg_similarity_to_gallery(features, identity, model_key='arcface'):
    """
    Compute average cosine similarity between gallery centroid and each condition.
    """
    gallery_vecs = np.array(features[identity]['gallery'][model_key])
    gallery_centroid = gallery_vecs.mean(axis=0)

    results = {}
    for condition in sorted(features[identity]['probe'].keys()):
        cond_vecs = features[identity]['probe'][condition][model_key]
        if len(cond_vecs) == 0:
            continue
        sims = [cosine_similarity(gallery_centroid, v) for v in cond_vecs]
        results[condition] = np.mean(sims)
    return results


print(f'\nAverage cosine similarity to gallery centroid:')
for identity in [id1, id2]:
    print(f'\n--- {identity} ---')
    print(f'{"Condition":<15} {"ArcFace":<12} {"VGG19":<12}')
    arc_sims = compute_avg_similarity_to_gallery(features, identity, 'arcface')
    vgg_sims = compute_avg_similarity_to_gallery(features, identity, 'vgg')
    for cond in sorted(arc_sims.keys()):
        print(f'{cond:<15} {arc_sims[cond]:<12.4f} {vgg_sims.get(cond, 0):<12.4f}')

### 3.5 Overall Summary

In [ ]:
print('=' * 50)
print('CLUSTERING ACCURACY SUMMARY')
print('=' * 50)
print(f'{"":<20} {"ArcFace":<12} {"VGG19":<12}')
print('-' * 44)
print(f'{"Clean Data":<20} {acc_arc_clean:<12.4f} {acc_vgg_clean:<12.4f}')
print(f'{"Noisy Data":<20} {acc_arc_noisy:<12.4f} {acc_vgg_noisy:<12.4f}')
print('-' * 44)
print(f'{"Drop (Clean→Noisy)":<20} {acc_arc_clean - acc_arc_noisy:<12.4f} {acc_vgg_clean - acc_vgg_noisy:<12.4f}')

### Discussion: Robustness of ArcFace vs VGG

**Write your discussion here:**

- How do varying conditions affect clustering performance?
- Which conditions are hardest for each model?
- Why is ArcFace expected to be more robust (trained with face-specific loss vs general ImageNet classification)?
- What do the PCA plots and cosine similarity numbers tell us about feature space structure?